# Stochastic Inventory Optimization Under Uncertain Demand

This notebook evaluates inventory decisions when demand is uncertain. All prices and demand probabilities are **illustrative assumptions for an academic simulation**, not real business data.

The model compares candidate stocking quantities using (1) exact expected profit across discrete scenarios and (2) Monte Carlo simulation.

## 1. Assumptions and demand scenarios
Selling price = ₹100, purchase cost = ₹60, salvage value = ₹10. Demand is 20, 40, or 60 units with probabilities 0.20, 0.50, and 0.30.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SELL_PRICE = 100
PURCHASE_COST = 60
SALVAGE_VALUE = 10
DEMAND_VALUES = np.array([20, 40, 60])
DEMAND_PROBS = np.array([0.20, 0.50, 0.30])
CANDIDATE_QUANTITIES = np.arange(0, 81, 5)
N_SIMULATIONS = 100_000
RANDOM_SEED = 42

pd.DataFrame({"Demand (units)": DEMAND_VALUES, "Probability": DEMAND_PROBS})

## 2. Profit model
Profit equals sales revenue plus salvage revenue from unsold units, minus the purchase cost.

In [ ]:
def calculate_profit(quantity, demand):
    sold = np.minimum(quantity, demand)
    unsold = np.maximum(quantity - demand, 0)
    return SELL_PRICE * sold + SALVAGE_VALUE * unsold - PURCHASE_COST * quantity

# Quick check: profit at Q=40 when demand is 20, 40, and 60
calculate_profit(40, DEMAND_VALUES)

## 3. Evaluate candidate quantities
For each stock quantity, calculate exact expected profit and estimate it using 100,000 simulated demand outcomes.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
simulated_demand = rng.choice(DEMAND_VALUES, size=N_SIMULATIONS, p=DEMAND_PROBS)

rows = []
for q in CANDIDATE_QUANTITIES:
    scenario_profit = calculate_profit(q, DEMAND_VALUES)
    simulated_profit = calculate_profit(q, simulated_demand)
    rows.append({
        "stock_quantity": int(q),
        "profit_demand_20": float(scenario_profit[0]),
        "profit_demand_40": float(scenario_profit[1]),
        "profit_demand_60": float(scenario_profit[2]),
        "expected_profit_inr": float(np.dot(scenario_profit, DEMAND_PROBS)),
        "simulated_expected_profit_inr": float(np.mean(simulated_profit)),
        "simulated_profit_std_inr": float(np.std(simulated_profit, ddof=1)),
        "simulated_profit_p05_inr": float(np.quantile(simulated_profit, 0.05)),
        "simulated_profit_p95_inr": float(np.quantile(simulated_profit, 0.95)),
    })

results = pd.DataFrame(rows)
results.to_csv("stock_quantity_results.csv", index=False)
results

In [ ]:
best = results.loc[results["expected_profit_inr"].idxmax()]
mean_demand = float(np.dot(DEMAND_VALUES, DEMAND_PROBS))
deterministic_quantity = int(round(mean_demand / 5) * 5)
baseline = results.loc[results["stock_quantity"] == deterministic_quantity].iloc[0]

print(f"Expected demand: {mean_demand:.1f} units")
print(f"Optimal stock quantity: {int(best['stock_quantity'])} units")
print(f"Maximum expected profit: ₹{best['expected_profit_inr']:.2f}")
print(f"Deterministic baseline quantity: {deterministic_quantity} units")
print(f"Baseline expected profit under uncertainty: ₹{baseline['expected_profit_inr']:.2f}")
print(f"Monte Carlo estimate at optimum: ₹{best['simulated_expected_profit_inr']:.2f}")

## 4. Expected-profit chart

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(results["stock_quantity"], results["expected_profit_inr"], marker="o", label="Exact expected profit")
plt.plot(results["stock_quantity"], results["simulated_expected_profit_inr"], linestyle="--", marker=".", label="Monte Carlo estimate")
plt.axvline(int(best["stock_quantity"]), linestyle=":", label="Optimal quantity")
plt.xlabel("Stock quantity (units)")
plt.ylabel("Expected profit (₹)")
plt.title("Expected Profit by Stock Quantity")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Sensitivity analysis
Keep medium-demand probability fixed at 0.50 and vary the probability of high demand. The low-demand probability adjusts so the probabilities still sum to 1.

In [ ]:
sensitivity_rows = []
for p_high in [0.10, 0.20, 0.30, 0.40, 0.50]:
    probs = np.array([0.50 - p_high, 0.50, p_high])
    expected = [
        float(np.dot(calculate_profit(q, DEMAND_VALUES), probs))
        for q in CANDIDATE_QUANTITIES
    ]
    i = int(np.argmax(expected))
    sensitivity_rows.append({
        "prob_demand_20": probs[0],
        "prob_demand_40": probs[1],
        "prob_demand_60": probs[2],
        "optimal_stock_quantity": int(CANDIDATE_QUANTITIES[i]),
        "maximum_expected_profit_inr": expected[i],
    })

sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv("sensitivity_analysis.csv", index=False)
sensitivity

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(sensitivity["prob_demand_60"], sensitivity["optimal_stock_quantity"], marker="o")
plt.xlabel("Probability of high demand (60 units)")
plt.ylabel("Optimal stock quantity (units)")
plt.title("Sensitivity to High-Demand Probability")
plt.tight_layout()
plt.show()

## 6. Interpretation and limitations

- The best quantity is the candidate with the largest expected profit under the assumed demand distribution.
- Monte Carlo estimates should be close to exact expected profits as the number of simulations increases.
- The deterministic baseline uses expected demand as if it were known with certainty; we evaluate that quantity against the same uncertain scenarios for a fair comparison.
- Results depend on the assumed prices, salvage value, demand values, and probabilities. These assumptions are illustrative and should not be treated as a forecast for a real retailer.